# Electricity Price Forecasting — DE-LU & ES  (Advanced LEAR Methods)
## Methods A (LEAR-ElasticNet), B (LightGBM-LEAR-v2), C (XGBoost-LEAR) + optimised ensemble
### Output: `predictions_v2.csv`

In [1]:
%pip install -q lightgbm xgboost statsmodels openmeteo-requests requests-cache retry-requests pandas numpy matplotlib seaborn scikit-learn scipy

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, json, warnings
from joblib import Parallel, delayed
import requests
import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize
from sklearn.linear_model import QuantileRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

try:
    import xgboost as xgb
    XGB_AVAILABLE = True
    print('XGBoost available:', xgb.__version__)
except ImportError:
    XGB_AVAILABLE = False
    print('XGBoost not available -- Method C will be skipped')

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (14, 5)})
RNG = np.random.default_rng(42)
print('Imports OK')

XGBoost available: 3.2.0
Imports OK


In [3]:
# Configuration
ZONES = {
    'DE-LU': {
        'ec_bzn':     'DE-LU',
        'ec_country': 'de',
        'lat': 51.1657,
        'lon': 10.4515,
        'label': 'Germany-Luxembourg',
    },
    'ES': {
        'ec_bzn':     'ES',
        'ec_country': 'es',
        'lat': 40.4168,
        'lon': -3.7038,
        'label': 'Spain (OMIE)',
    },
}

TRAIN_START         = '2022-01-01'
TRAIN_END           = '2026-05-09'
WEATHER_ARCHIVE_END = '2026-05-04'
CUTOFF_DAYS         = 14
QUANTILES           = [0.025, 0.45, 0.975]
DATA_DIR            = './data/'

# Evaluation window: Mon 11 May 02:00 CEST -> Tue 12 May 01:00 CEST (24 rows)
# CEST = UTC+1, so 02:00 CEST = 01:00 UTC
EVAL_TIMESTAMPS = pd.date_range(
    start='2026-05-11 01:00', periods=24, freq='1h', tz='UTC'
)
MAY10_TIMESTAMPS = pd.date_range(
    start='2026-05-10 00:00', periods=24, freq='1h', tz='UTC'
)

os.makedirs(DATA_DIR, exist_ok=True)
print('Evaluation window:', EVAL_TIMESTAMPS[0], '->', EVAL_TIMESTAMPS[-1])
print('Slots:', len(EVAL_TIMESTAMPS))

Evaluation window: 2026-05-11 01:00:00+00:00 -> 2026-05-12 00:00:00+00:00
Slots: 24


In [4]:
# Energy-Charts + Open-Meteo API helpers (cached to CSV)
EC_BASE = 'https://api.energy-charts.info'


def ec_prices(bzn, start, end, cache_path):
    if os.path.exists(cache_path):
        df = pd.read_csv(cache_path, index_col=0, parse_dates=True)
        df.index = pd.to_datetime(df.index, utc=True)
        return df
    r = requests.get(
        f'{EC_BASE}/price',
        params={'bzn': bzn, 'start': start, 'end': end},
        timeout=120,
    )
    r.raise_for_status()
    d = r.json()
    ts = pd.to_datetime(d['unix_seconds'], unit='s', utc=True)
    df = pd.DataFrame({'price': d['price']}, index=ts)
    df.index.name = 'timestamp'
    df.to_csv(cache_path)
    return df


def ec_generation(country, start, end, cache_path):
    if os.path.exists(cache_path):
        df = pd.read_csv(cache_path, index_col=0, parse_dates=True)
        df.index = pd.to_datetime(df.index, utc=True)
        return df
    r = requests.get(
        f'{EC_BASE}/public_power',
        params={'country': country, 'start': start, 'end': end},
        timeout=180,
    )
    r.raise_for_status()
    d = r.json()
    ts = pd.to_datetime(d['unix_seconds'], unit='s', utc=True)
    df = pd.DataFrame(index=ts)
    df.index.name = 'timestamp'
    for series in d.get('production_types', []):
        col = series['name'].lower().replace(' ', '_').replace('-', '_')
        df[col] = series.get('data', [None] * len(ts))
    df.to_csv(cache_path)
    return df


def om_weather(lat, lon, start, end, cache_path, forecast=False):
    if os.path.exists(cache_path):
        df = pd.read_csv(cache_path, index_col=0, parse_dates=True)
        df.index = pd.to_datetime(df.index, utc=True)
        return df
    variables = 'temperature_2m,wind_speed_10m,shortwave_radiation'
    if forecast:
        url    = 'https://api.open-meteo.com/v1/forecast'
        params = {
            'latitude': lat, 'longitude': lon, 'hourly': variables,
            'timezone': 'UTC', 'forecast_days': 16, 'past_days': 16,
        }
    else:
        url    = 'https://archive-api.open-meteo.com/v1/archive'
        params = {
            'latitude': lat, 'longitude': lon, 'start_date': start,
            'end_date': end, 'hourly': variables, 'timezone': 'UTC',
        }
    r = requests.get(url, params=params, timeout=120)
    r.raise_for_status()
    d = r.json()['hourly']
    df = pd.DataFrame({
        'temperature':     d['temperature_2m'],
        'wind_speed':      d['wind_speed_10m'],
        'solar_radiation': d['shortwave_radiation'],
    }, index=pd.to_datetime(d['time'], utc=True))
    df.index.name = 'timestamp'
    df.to_csv(cache_path)
    return df


print('API helpers defined.')

API helpers defined.


In [5]:
# Load data (reuses cached CSVs from model.ipynb)
raw = {}
for zone, cfg in ZONES.items():
    slug = zone.lower().replace('-', '_')
    print(f'Loading {zone}...')
    raw[zone] = {}
    raw[zone]['prices'] = ec_prices(
        cfg['ec_bzn'], TRAIN_START, TRAIN_END,
        f"{DATA_DIR}{slug}_prices.csv"
    )
    raw[zone]['gen'] = ec_generation(
        cfg['ec_country'], TRAIN_START, TRAIN_END,
        f"{DATA_DIR}{slug}_generation.csv"
    )
    raw[zone]['weather_hist'] = om_weather(
        cfg['lat'], cfg['lon'], TRAIN_START, WEATHER_ARCHIVE_END,
        f"{DATA_DIR}{slug}_weather_hist.csv"
    )
    raw[zone]['weather_fcst'] = om_weather(
        cfg['lat'], cfg['lon'], None, None,
        f"{DATA_DIR}{slug}_weather_fcst.csv",
        forecast=True,
    )
    print(f"  prices: {len(raw[zone]['prices'])} | "
          f"gen: {len(raw[zone]['gen'])} | "
          f"wx_hist: {len(raw[zone]['weather_hist'])}")

print('All data loaded.')

Loading DE-LU...
  prices: 54071 | gen: 152563 | wx_hist: 38040
Loading ES...
  prices: 54071 | gen: 126278 | wx_hist: 38040
All data loaded.


In [6]:
# Preprocessing

def compute_renewable_share(gen_df):
    cols       = gen_df.columns.tolist()
    wind_cols  = [c for c in cols if 'wind' in c]
    solar_cols = [c for c in cols if 'solar' in c or 'photovoltaic' in c]
    total_cols = [c for c in cols if 'total' in c or 'load' in c]
    renew = gen_df[wind_cols + solar_cols].clip(lower=0).sum(axis=1)
    total = (
        gen_df[total_cols].clip(lower=0).sum(axis=1)
        if total_cols else renew * 2
    )
    share = (renew / total.replace(0, np.nan)).ffill().clip(0, 1)
    return share.rename('renewable_share')


def merge_zone(zone):
    prices  = raw[zone]['prices'].copy()
    gen     = raw[zone]['gen'].copy()
    wx_hist = raw[zone]['weather_hist'].copy()
    wx_fcst = raw[zone]['weather_fcst'].copy()

    wx = pd.concat([wx_hist, wx_fcst]).sort_index()
    wx = wx[~wx.index.duplicated(keep='last')]
    rs = compute_renewable_share(gen)

    df = prices.join(wx, how='outer').join(rs, how='outer')
    df = df.resample('1h').asfreq()
    df = df.ffill(limit=3)
    df['price'] = df['price'].clip(lower=-500, upper=3000)
    df = df.fillna(df.median(numeric_only=True))
    print(f'{zone}: {len(df)} rows, {df.isna().sum().sum()} NaNs remaining')
    return df


merged = {z: merge_zone(z) for z in ZONES}

DE-LU: 38521 rows, 0 NaNs remaining
ES: 38521 rows, 0 NaNs remaining


In [7]:
# LEAR Feature Engineering  (Lago et al. 2021)
#
# For hour h, target day D:
#   d1_h{j} = price at (D-1) 00:00 + j hours
#             computed as: df['price'].shift(h + 24 - j)  on the full series,
#             then masked to rows where index.hour == h
#   d7_h{j} = price at (D-7) 00:00 + j hours
#             computed as: df['price'].shift(h + 168 - j) on the full series
#
# Example correctness check: h=10, j=0 -> shift(34)
#   34 hours before 10:00 UTC = previous day 00:00 UTC  (correct)
#
# Features per hour: 24 (D-1 profile) + 24 (D-7 profile) + 14 scalars = 62

LEAR_SCALAR_COLS = [
    'd1_min', 'd1_max', 'd1_mean',
    'temperature', 'wind_speed', 'solar_radiation', 'solar_hour',
    'renewable_share',
    'dow_sin', 'dow_cos', 'month_sin', 'month_cos',
    'is_weekend', 'is_monday',
]

LEAR_LAG_COLS = (
    [f'd1_h{j:02d}' for j in range(24)] +
    [f'd7_h{j:02d}' for j in range(24)]
)

LEAR_FEATURE_COLS = LEAR_LAG_COLS + LEAR_SCALAR_COLS
print(f'LEAR feature count: {len(LEAR_FEATURE_COLS)}')


def build_lear_features(df, hour):
    mask   = df.index.hour == hour
    h_df   = df[mask].copy()
    idx    = h_df.index
    X      = pd.DataFrame(index=idx)
    p_full = df['price']

    # D-1 full price profile (24 lag features)
    for j in range(24):
        X[f'd1_h{j:02d}'] = p_full.shift(hour + 24 - j)[mask].values

    # D-7 full price profile (24 lag features)
    for j in range(24):
        X[f'd7_h{j:02d}'] = p_full.shift(hour + 168 - j)[mask].values

    # D-1 summary stats derived from lag columns
    d1_cols = [f'd1_h{j:02d}' for j in range(24)]
    X['d1_min']  = X[d1_cols].min(axis=1)
    X['d1_max']  = X[d1_cols].max(axis=1)
    X['d1_mean'] = X[d1_cols].mean(axis=1)

    # Weather
    solar = h_df.get('solar_radiation', pd.Series(0.0, index=idx))
    X['temperature']     = h_df.get('temperature',  pd.Series(15.0, index=idx))
    X['wind_speed']      = h_df.get('wind_speed',   pd.Series(4.0,  index=idx))
    X['solar_radiation'] = solar
    X['solar_hour']      = solar * np.abs(np.cos(2 * np.pi * hour / 24))
    X['renewable_share'] = h_df.get('renewable_share', pd.Series(0.3, index=idx))

    # Calendar encoding
    X['dow_sin']    = np.sin(2 * np.pi * idx.dayofweek / 7)
    X['dow_cos']    = np.cos(2 * np.pi * idx.dayofweek / 7)
    X['month_sin']  = np.sin(2 * np.pi * idx.month / 12)
    X['month_cos']  = np.cos(2 * np.pi * idx.month / 12)
    X['is_weekend'] = (idx.dayofweek >= 5).astype(int)
    X['is_monday']  = (idx.dayofweek == 0).astype(int)

    return X[LEAR_FEATURE_COLS]


# Build 24 x 2 LEAR feature matrices
print('Building LEAR feature matrices...')
lear_features = {
    zone: {h: build_lear_features(merged[zone], h) for h in range(24)}
    for zone in ZONES
}

for zone in ZONES:
    ex = lear_features[zone][12]
    print(f'  {zone} h=12: shape={ex.shape}, NaN rows={ex.isna().any(axis=1).sum()}')

print('LEAR feature matrices built.')

LEAR feature count: 62
Building LEAR feature matrices...
  DE-LU h=12: shape=(1605, 62), NaN rows=7
  ES h=12: shape=(1605, 62), NaN rows=7
LEAR feature matrices built.


In [8]:
# Pinball loss
def pinball(y_true, y_pred, q):
    r = np.asarray(y_true) - np.asarray(y_pred)
    return float(np.mean(np.where(r >= 0, q * r, (q - 1) * r)))


# Method A: LEAR-ElasticNet — parallelised over hours
def _train_elasticnet_hour(h, X_h, y_h):
    valid = X_h.notna().all(axis=1) & y_h.notna()
    X_v, y_v = X_h[valid], y_h[valid]
    split = len(X_v) - 56
    X_tr, X_val = X_v.iloc[:split], X_v.iloc[split:]
    y_tr, y_val = y_v.iloc[:split], y_v.iloc[split:]
    models = {}
    val_preds = {'y_val': y_val}
    for q in QUANTILES:
        pipe = Pipeline([
            ('scaler', StandardScaler()),
            ('qr', QuantileRegressor(quantile=q, alpha=0.1, solver='highs')),
        ])
        pipe.fit(X_tr.values, y_tr.values)
        models[q] = pipe
        val_preds[q] = pipe.predict(X_val.values)
    return h, models, val_preds


def train_lear_elasticnet(zone):
    results = Parallel(n_jobs=-1, prefer='threads')(  # threads safe for sklearn
        delayed(_train_elasticnet_hour)(
            h,
            lear_features[zone][h],
            merged[zone]['price'].loc[lear_features[zone][h].index],
        )
        for h in range(24)
    )
    models_out = {h: m for h, m, _ in results}
    probe_out  = {h: p for h, _, p in results}
    return models_out, probe_out


print('Training LEAR-ElasticNet DE-LU...')
models_a_delu, probe_a_delu = train_lear_elasticnet('DE-LU')
print('Training LEAR-ElasticNet ES...')
models_a_es, probe_a_es = train_lear_elasticnet('ES')
models_a = {'DE-LU': models_a_delu, 'ES': models_a_es}
probe_a  = {'DE-LU': probe_a_delu,  'ES': probe_a_es}
print('Method A complete.')

Training LEAR-ElasticNet DE-LU...
Training LEAR-ElasticNet ES...
Method A complete.


In [9]:
# Method B: LightGBM-LEAR-v2 — parallelised over hours
# n_jobs=1 per model avoids core oversubscription when 24 run in parallel
LGBM_LEAR_PARAMS = dict(
    n_estimators      = 600,
    learning_rate     = 0.04,
    num_leaves        = 63,
    min_child_samples = 20,
    subsample         = 0.8,
    colsample_bytree  = 0.7,
    reg_alpha         = 0.05,
    reg_lambda        = 1.0,
    n_jobs            = 1,   # 1 per model; outer Parallel uses all cores
    verbose           = -1,
)


def _train_lgbm_hour(h, X_h, y_h):
    valid = X_h.notna().all(axis=1) & y_h.notna()
    X_v, y_v = X_h[valid], y_h[valid]
    split = len(X_v) - 56
    X_tr, X_vl = X_v.iloc[:split], X_v.iloc[split:]
    y_tr, y_vl = y_v.iloc[:split], y_v.iloc[split:]
    models = {}
    val_preds = {'y_val': y_vl}
    for q in QUANTILES:
        base_p = {**LGBM_LEAR_PARAMS, 'objective': 'quantile', 'alpha': q}
        m_probe = lgb.LGBMRegressor(**base_p)
        m_probe.fit(X_tr, y_tr, eval_set=[(X_vl, y_vl)],
                    callbacks=[lgb.early_stopping(50, verbose=False),
                               lgb.log_evaluation(0)])
        best_iter = m_probe.best_iteration_
        val_preds[q] = m_probe.predict(X_vl)
        m_final = lgb.LGBMRegressor(**{**base_p, 'n_estimators': best_iter})
        m_final.fit(X_v, y_v)
        models[q] = m_final
    return h, models, val_preds


def train_lgbm_lear(zone):
    results = Parallel(n_jobs=-1, prefer='threads')(
        delayed(_train_lgbm_hour)(
            h,
            lear_features[zone][h],
            merged[zone]['price'].loc[lear_features[zone][h].index],
        )
        for h in range(24)
    )
    models_out = {h: m for h, m, _ in results}
    probe_out  = {h: p for h, _, p in results}
    return models_out, probe_out


print('Training LightGBM-LEAR-v2 DE-LU...')
models_b_delu, probe_b_delu = train_lgbm_lear('DE-LU')
print('Training LightGBM-LEAR-v2 ES...')
models_b_es, probe_b_es = train_lgbm_lear('ES')
models_b = {'DE-LU': models_b_delu, 'ES': models_b_es}
probe_b  = {'DE-LU': probe_b_delu,  'ES': probe_b_es}
print('Method B complete.')

Training LightGBM-LEAR-v2 DE-LU...
Training LightGBM-LEAR-v2 ES...
Method B complete.


In [10]:
# Method C: XGBoost-LEAR — parallelised over hours
models_c = None
probe_c  = None

if XGB_AVAILABLE:
    try:
        _test = xgb.XGBRegressor(objective='reg:quantileerror',
                                  quantile_alpha=0.5, n_estimators=2, verbosity=0)
        _test.fit([[0]*62], [0])
        XGB_COMPAT = True
    except Exception as e:
        print('XGBoost quantile objective unavailable:', e)
        XGB_COMPAT = False
else:
    XGB_COMPAT = False

if XGB_COMPAT:
    XGB_PARAMS = dict(
        objective        = 'reg:quantileerror',
        n_estimators     = 500,
        learning_rate    = 0.04,
        max_depth        = 6,
        subsample        = 0.8,
        colsample_bytree = 0.7,
        reg_alpha        = 0.05,
        reg_lambda       = 1.0,
        n_jobs           = 1,   # 1 per model; outer Parallel uses all cores
        verbosity        = 0,
        random_state     = 42,
    )

    def _train_xgb_hour(h, X_h, y_h):
        valid = X_h.notna().all(axis=1) & y_h.notna()
        X_v, y_v = X_h[valid], y_h[valid]
        split = len(X_v) - 56
        X_tr, X_vl = X_v.iloc[:split], X_v.iloc[split:]
        y_tr, y_vl = y_v.iloc[:split], y_v.iloc[split:]
        models = {}
        val_preds = {'y_val': y_vl}
        for q in QUANTILES:
            m = xgb.XGBRegressor(**{**XGB_PARAMS, 'quantile_alpha': q})
            m.fit(X_tr, y_tr, eval_set=[(X_vl, y_vl)], verbose=False)
            val_preds[q] = m.predict(X_vl)
            m_full = xgb.XGBRegressor(**{**XGB_PARAMS, 'quantile_alpha': q})
            m_full.fit(X_v, y_v)
            models[q] = m_full
        return h, models, val_preds

    def train_xgb_lear(zone):
        results = Parallel(n_jobs=-1, prefer='threads')(
            delayed(_train_xgb_hour)(
                h,
                lear_features[zone][h],
                merged[zone]['price'].loc[lear_features[zone][h].index],
            )
            for h in range(24)
        )
        return {h: m for h, m, _ in results}, {h: p for h, _, p in results}

    print('Training XGBoost-LEAR DE-LU...')
    _mc_delu, _pc_delu = train_xgb_lear('DE-LU')
    print('Training XGBoost-LEAR ES...')
    _mc_es, _pc_es = train_xgb_lear('ES')
    models_c = {'DE-LU': _mc_delu, 'ES': _mc_es}
    probe_c  = {'DE-LU': _pc_delu, 'ES': _pc_es}
    print('Method C complete.')
else:
    print('Method C skipped.')

Training XGBoost-LEAR DE-LU...
Training XGBoost-LEAR ES...
Method C complete.


In [11]:
# Validation Comparison Table
# Pinball(q=0.45) on last 8 weeks held-out validation set per zone and hour.

print('Validation pinball(q=0.45) -- last 8 weeks held out')
print('-' * 60)

summary = {zone: {'A': [], 'B': [], 'C': []} for zone in ZONES}

for zone in ZONES:
    print(f'Zone: {zone}')
    for h in range(24):
        y_val = probe_a[zone][h]['y_val']
        pb_a  = pinball(y_val, probe_a[zone][h][0.45], 0.45)
        pb_b  = pinball(probe_b[zone][h]['y_val'], probe_b[zone][h][0.45], 0.45)
        summary[zone]['A'].append(pb_a)
        summary[zone]['B'].append(pb_b)

        pb_c_str = 'N/A  '
        if probe_c is not None and probe_c[zone] is not None:
            pb_c = pinball(probe_c[zone][h]['y_val'], probe_c[zone][h][0.45], 0.45)
            summary[zone]['C'].append(pb_c)
            pb_c_str = f'{pb_c:.4f}'

        if h % 6 == 0:
            print(f'  h={h:02d}  A={pb_a:.4f}  B={pb_b:.4f}  C={pb_c_str}')

print()
print('Mean pinball across all hours:')
for zone in ZONES:
    a_mean = np.mean(summary[zone]['A'])
    b_mean = np.mean(summary[zone]['B'])
    c_str  = f"{np.mean(summary[zone]['C']):.4f}" if summary[zone]['C'] else 'N/A'
    print(f'  {zone}: A={a_mean:.4f}  B={b_mean:.4f}  C={c_str}')

Validation pinball(q=0.45) -- last 8 weeks held out
------------------------------------------------------------
Zone: DE-LU
  h=00  A=1.6973  B=2.0239  C=2.3933
  h=06  A=10.5768  B=9.3963  C=9.2605
  h=12  A=21.1461  B=21.1571  C=19.9351
  h=18  A=13.0871  B=10.9263  C=11.5413
Zone: ES
  h=00  A=1.8542  B=1.8625  C=1.8438
  h=06  A=6.2525  B=5.7360  C=5.7459
  h=12  A=5.9618  B=3.6871  C=4.1469
  h=18  A=9.4467  B=8.8485  C=8.7793

Mean pinball across all hours:
  DE-LU: A=9.4752  B=8.4873  C=8.6660
  ES: A=6.0334  B=5.0677  C=5.3500


In [12]:
# Optimise Convex Ensemble Weights
# For each (zone, hour, quantile): find w_A, w_B, [w_C] >= 0 summing to 1
# that minimise validation pinball loss via SLSQP.


def optimise_weights(preds_list, y_val_arr, q):
    n = len(preds_list)
    if n == 1:
        return np.array([1.0])

    def objective(w):
        ens = sum(w[i] * preds_list[i] for i in range(n))
        return pinball(y_val_arr, ens, q)

    w0 = np.ones(n) / n
    constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}]
    bounds = [(0, 1)] * n
    result = minimize(
        objective, w0, method='SLSQP',
        bounds=bounds, constraints=constraints,
        options={'ftol': 1e-9, 'maxiter': 200},
    )
    return result.x if result.success else w0


# ensemble_weights[zone][h][q] = weight array (length = number of methods)
ensemble_weights = {}

for zone in ZONES:
    ensemble_weights[zone] = {}
    for h in range(24):
        ensemble_weights[zone][h] = {}
        y_val = probe_a[zone][h]['y_val'].values

        for q in QUANTILES:
            preds_list = [
                probe_a[zone][h][q],
                probe_b[zone][h][q],
            ]
            if probe_c is not None and probe_c[zone] is not None:
                preds_list.append(probe_c[zone][h][q])

            w = optimise_weights(preds_list, y_val, q)
            ensemble_weights[zone][h][q] = w

print('Average ensemble weights across all hours:')
for zone in ZONES:
    for q in QUANTILES:
        all_w  = np.array([ensemble_weights[zone][h][q] for h in range(24)])
        mean_w = all_w.mean(axis=0)
        labels = ['A', 'B', 'C'][:len(mean_w)]
        w_str  = '  '.join(f'{lbl}={v:.3f}' for lbl, v in zip(labels, mean_w))
        print(f'  {zone} q={q:.3f}: {w_str}')

Average ensemble weights across all hours:
  DE-LU q=0.025: A=0.013  B=0.903  C=0.084
  DE-LU q=0.450: A=0.227  B=0.455  C=0.317
  DE-LU q=0.975: A=0.042  B=0.228  C=0.730
  ES q=0.025: A=0.087  B=0.706  C=0.207
  ES q=0.450: A=0.225  B=0.570  C=0.205
  ES q=0.975: A=0.051  B=0.150  C=0.799


In [13]:
# LEAR Evaluation Feature Builder
# Same formula as build_lear_features but operates on an extended DataFrame
# (historical + bridge + future rows) and filters to specific target_timestamps.


def build_lear_eval_features(extended_df, hour, target_timestamps):
    mask   = extended_df.index.hour == hour
    h_df   = extended_df[mask].copy()
    idx    = h_df.index
    X_full = pd.DataFrame(index=idx)
    p_full = extended_df['price']

    # D-1 full price profile
    for j in range(24):
        X_full[f'd1_h{j:02d}'] = p_full.shift(hour + 24 - j)[mask].values

    # D-7 full price profile
    for j in range(24):
        X_full[f'd7_h{j:02d}'] = p_full.shift(hour + 168 - j)[mask].values

    d1_cols = [f'd1_h{j:02d}' for j in range(24)]
    X_full['d1_min']  = X_full[d1_cols].min(axis=1)
    X_full['d1_max']  = X_full[d1_cols].max(axis=1)
    X_full['d1_mean'] = X_full[d1_cols].mean(axis=1)

    solar = h_df.get('solar_radiation', pd.Series(0.0, index=idx))
    X_full['temperature']     = h_df.get('temperature',  pd.Series(15.0, index=idx))
    X_full['wind_speed']      = h_df.get('wind_speed',   pd.Series(4.0,  index=idx))
    X_full['solar_radiation'] = solar
    X_full['solar_hour']      = solar * np.abs(np.cos(2 * np.pi * hour / 24))
    X_full['renewable_share'] = h_df.get('renewable_share', pd.Series(0.3, index=idx))

    X_full['dow_sin']    = np.sin(2 * np.pi * idx.dayofweek / 7)
    X_full['dow_cos']    = np.cos(2 * np.pi * idx.dayofweek / 7)
    X_full['month_sin']  = np.sin(2 * np.pi * idx.month / 12)
    X_full['month_cos']  = np.cos(2 * np.pi * idx.month / 12)
    X_full['is_weekend'] = (idx.dayofweek >= 5).astype(int)
    X_full['is_monday']  = (idx.dayofweek == 0).astype(int)

    X_full = X_full[LEAR_FEATURE_COLS]

    available = [ts for ts in target_timestamps if ts in X_full.index]
    if not available:
        return pd.DataFrame(columns=LEAR_FEATURE_COLS)
    return X_full.loc[available]


print('build_lear_eval_features defined.')

build_lear_eval_features defined.


In [14]:
# Bridge Step: Predict May 10, Build merged_extended, Compute LEAR Eval Features
#
# Two-step prediction strategy:
#   Step 1+2: Predict May 10 prices using Method B (LEAR features from history)
#   Step 3:   Build merged_extended = history + May10 bridge + May11 eval shell
#   Step 4:   Compute LEAR features for May 11 using May 10 as the D-1 profile

# Step 1: Build LEAR features for May 10 from historical data
print('Step 1: Building LEAR features for May 10 bridge...')
may10_lear_feats = {}
for zone in ZONES:
    may10_lear_feats[zone] = {}
    for h in range(24):
        ts_h = [ts for ts in MAY10_TIMESTAMPS if ts.hour == h]
        if ts_h:
            may10_lear_feats[zone][h] = build_lear_eval_features(
                merged[zone], h, ts_h
            )

# Step 2: Predict May 10 with Method B (LightGBM-LEAR-v2)
print('Step 2: Predicting May 10 prices with LightGBM-LEAR-v2...')
may10_preds = {}
for zone in ZONES:
    preds = []
    for ts in MAY10_TIMESTAMPS:
        h = ts.hour
        feat_h = may10_lear_feats[zone].get(h)
        if (
            feat_h is not None
            and ts in feat_h.index
            and not feat_h.loc[[ts]].isna().any().any()
        ):
            p50 = float(models_b[zone][h][0.45].predict(feat_h.loc[[ts]])[0])
        else:
            p50 = float(merged[zone]['price'].dropna().iloc[-1])
            print(f'  Warning: using fallback for {zone} {ts}')
        preds.append((ts, p50))
    may10_preds[zone] = pd.Series(dict(preds))
    lo = may10_preds[zone].min()
    hi = may10_preds[zone].max()
    print(f'  {zone} May 10 bridge range: [{lo:.1f}, {hi:.1f}] EUR/MWh')

# Step 3: Build merged_extended
print('Step 3: Building merged_extended...')
merged_extended = {}
for zone in ZONES:
    df_hist = merged[zone].copy()
    wx_fcst = raw[zone]['weather_fcst']

    all_future_ts = MAY10_TIMESTAMPS.append(EVAL_TIMESTAMPS)
    future_df = pd.DataFrame(index=all_future_ts)
    future_df['price'] = np.nan

    # Inject bridge predictions so D-1 lags are populated for May 11
    for ts, p in may10_preds[zone].items():
        if ts in future_df.index:
            future_df.loc[ts, 'price'] = p

    for col in ['temperature', 'wind_speed', 'solar_radiation']:
        future_df[col] = wx_fcst[col].reindex(all_future_ts)

    rs_profile = (
        df_hist['renewable_share']
        .groupby([df_hist.index.month, df_hist.index.hour])
        .mean()
    )
    future_df['renewable_share'] = [
        rs_profile.get((ts.month, ts.hour), 0.3)
        for ts in all_future_ts
    ]

    combined = pd.concat([df_hist, future_df])
    combined = combined[~combined.index.duplicated(keep='last')].sort_index()
    merged_extended[zone] = combined
    print(f'  {zone}: extended df = {len(combined)} rows')

# Step 4: Build LEAR eval features for May 11
print('Step 4: Building LEAR eval features for May 11...')
lear_eval_feats = {}
for zone in ZONES:
    lear_eval_feats[zone] = {}
    n_clean = 0
    for h in range(24):
        ts_h = [ts for ts in EVAL_TIMESTAMPS if ts.hour == h]
        if ts_h:
            feat_h = build_lear_eval_features(merged_extended[zone], h, ts_h)
            lear_eval_feats[zone][h] = feat_h
            if len(feat_h) > 0 and not feat_h.isna().any().any():
                n_clean += 1
    print(f'  {zone}: {n_clean}/24 eval hour slots NaN-free')

Step 1: Building LEAR features for May 10 bridge...
Step 2: Predicting May 10 prices with LightGBM-LEAR-v2...
  DE-LU May 10 bridge range: [60.3, 127.7] EUR/MWh
  ES May 10 bridge range: [4.3, 89.1] EUR/MWh
Step 3: Building merged_extended...
  DE-LU: extended df = 38521 rows
  ES: extended df = 38521 rows
Step 4: Building LEAR eval features for May 11...
  DE-LU: 23/24 eval hour slots NaN-free
  ES: 23/24 eval hour slots NaN-free


In [15]:
def predict_advanced(ts, zone):
    h = ts.hour
    feat_h = lear_eval_feats[zone].get(h)
    has_valid = (
        feat_h is not None
        and ts in feat_h.index
        and not feat_h.loc[[ts]].isna().any().any()
    )

    if not has_valid:
        # Fallback: use recent 90-day price quantiles (avoids 2022 crisis contamination)
        recent = merged[zone]['price'].dropna().iloc[-90 * 24:]
        out = {q: float(recent.quantile(q)) for q in QUANTILES}
        p025, p50, p975 = sorted([out[0.025], out[0.45], out[0.975]])
        return {'p025': p025, 'p50': p50, 'p975': p975, 'regime': 'fallback_recent'}

    x = feat_h.loc[[ts]]
    row_out = {}
    for q in QUANTILES:
        preds_list = [
            float(models_a[zone][h][q].predict(x.values)[0]),
            float(models_b[zone][h][q].predict(x)[0]),
        ]
        if models_c is not None and models_c[zone] is not None:
            preds_list.append(float(models_c[zone][h][q].predict(x)[0]))

        w_q = ensemble_weights[zone][h][q][:len(preds_list)]
        w_q = w_q / w_q.sum()
        row_out[q] = float(np.dot(w_q, preds_list))

    p025, p50, p975 = sorted([row_out[0.025], row_out[0.45], row_out[0.975]])
    return {'p025': p025, 'p50': p50, 'p975': p975, 'regime': 'ensemble'}


print('Running predictions for evaluation window (May 11 UTC)...')
eval_preds_v2 = {}
for zone in ZONES:
    results = [predict_advanced(ts, zone) for ts in EVAL_TIMESTAMPS]
    eval_preds_v2[zone] = pd.DataFrame(results, index=EVAL_TIMESTAMPS)

for zone in ZONES:
    print(zone)
    print(eval_preds_v2[zone][['p025', 'p50', 'p975']].round(2).to_string())
    print()

Running predictions for evaluation window (May 11 UTC)...
DE-LU
                             p025     p50    p975
2026-05-11 01:00:00+00:00   90.99   98.27  112.00
2026-05-11 02:00:00+00:00   88.45   98.13  120.65
2026-05-11 03:00:00+00:00   93.90  103.97  115.50
2026-05-11 04:00:00+00:00  101.10  113.05  142.25
2026-05-11 05:00:00+00:00  105.45  126.94  199.54
2026-05-11 06:00:00+00:00  105.44  124.78  209.62
2026-05-11 07:00:00+00:00   84.66  112.60  148.60
2026-05-11 08:00:00+00:00   47.93   98.02  101.04
2026-05-11 09:00:00+00:00   23.33   85.56  108.16
2026-05-11 10:00:00+00:00   20.44   72.58   97.51
2026-05-11 11:00:00+00:00    5.89   63.91   86.46
2026-05-11 12:00:00+00:00    3.33   58.02   80.80
2026-05-11 13:00:00+00:00    9.86   60.60   84.17
2026-05-11 14:00:00+00:00   14.71   79.65   88.09
2026-05-11 15:00:00+00:00   45.84   91.28  102.71
2026-05-11 16:00:00+00:00   84.50  103.97  143.72
2026-05-11 17:00:00+00:00   98.90  119.93  170.61
2026-05-11 18:00:00+00:00   94.59  1

In [16]:
# Sanity Checks
for zone in ZONES:
    df_p = eval_preds_v2[zone]

    mono_ok = (
        (df_p['p025'] <= df_p['p50']) &
        (df_p['p50']  <= df_p['p975'])
    ).all()
    assert mono_ok, f'Quantile monotonicity violated for {zone}'
    assert not df_p.isna().any().any(), f'NaN values in predictions for {zone}'
    assert len(df_p) == 24, f'Expected 24 rows for {zone}, got {len(df_p)}'

    p50_rng = f'[{df_p["p50"].min():.1f}, {df_p["p50"].max():.1f}]'
    print(f'{zone}: OK -- 24 rows, monotone, p50 range = {p50_rng} EUR/MWh')

for zone in ZONES:
    regime_counts = eval_preds_v2[zone]['regime'].value_counts()
    print(f'{zone} regimes: {dict(regime_counts)}')

print('All sanity checks passed.')

DE-LU: OK -- 24 rows, monotone, p50 range = [58.0, 138.6] EUR/MWh
ES: OK -- 24 rows, monotone, p50 range = [29.9, 103.7] EUR/MWh
DE-LU regimes: {'ensemble': 23, 'fallback_recent': 1}
ES regimes: {'ensemble': 23, 'fallback_recent': 1}
All sanity checks passed.


In [17]:
# Save predictions_v2.csv
# Format: UTC timestamps formatted as CET (+01:00), matching competition spec.


def format_cet(ts_utc):
    cet = ts_utc + pd.Timedelta(hours=1)
    return cet.strftime('%Y-%m-%dT%H:%M:%S+01:00')


rows = []
for ts in EVAL_TIMESTAMPS:
    de = eval_preds_v2['DE-LU'].loc[ts]
    es = eval_preds_v2['ES'].loc[ts]
    rows.append({
        'timestamp':  format_cet(ts),
        'DE-LU p025': round(float(de['p025']), 2),
        'DE-LU p50':  round(float(de['p50']),  2),
        'DE-LU p975': round(float(de['p975']), 2),
        'ES p025':    round(float(es['p025']), 2),
        'ES p50':     round(float(es['p50']),  2),
        'ES p975':    round(float(es['p975']), 2),
    })

out_df = pd.DataFrame(rows)
out_df.to_csv('predictions_v2.csv', index=False)

print(f'Saved {len(out_df)} rows to predictions_v2.csv')
print()
print(out_df.to_string(index=False))

Saved 24 rows to predictions_v2.csv

                timestamp  DE-LU p025  DE-LU p50  DE-LU p975  ES p025  ES p50  ES p975
2026-05-11T02:00:00+01:00       90.99      98.27      112.00    53.03   79.19    93.80
2026-05-11T03:00:00+01:00       88.45      98.13      120.65    58.44   75.99    90.38
2026-05-11T04:00:00+01:00       93.90     103.97      115.50    58.75   75.98    98.28
2026-05-11T05:00:00+01:00      101.10     113.05      142.25    56.62   79.79   106.63
2026-05-11T06:00:00+01:00      105.45     126.94      199.54    73.91   91.36   117.81
2026-05-11T07:00:00+01:00      105.44     124.78      209.62    52.49   92.75   147.62
2026-05-11T08:00:00+01:00       84.66     112.60      148.60    33.63   77.31   119.15
2026-05-11T09:00:00+01:00       47.93      98.02      101.04    12.71   59.32    89.62
2026-05-11T10:00:00+01:00       23.33      85.56      108.16     3.79   49.25    83.24
2026-05-11T11:00:00+01:00       20.44      72.58       97.51     1.61   41.12    69.53
2026-0

In [18]:
# Comparison: predictions_v2.csv vs predictions.csv

if not os.path.exists('predictions.csv'):
    print('predictions.csv not found -- skipping comparison.')
else:
    v1 = pd.read_csv('predictions.csv')
    v2 = pd.read_csv('predictions_v2.csv')

    price_cols = [
        'DE-LU p025', 'DE-LU p50', 'DE-LU p975',
        'ES p025', 'ES p50', 'ES p975',
    ]

    print('Column-wise mean absolute difference (v2 - v1):')
    print('-' * 50)
    for col in price_cols:
        diff = (v2[col] - v1[col]).abs().mean()
        net  = (v2[col] - v1[col]).mean()
        sign = '+' if net >= 0 else ''
        print(f'  {col:<14s}: MAD={diff:.2f}  net_bias={sign}{net:.2f}')

    print()
    print('Row-wise p50 comparison (v1 vs v2):')
    print('-' * 72)
    hdr = (
        f"{'Timestamp':<28s} "
        f"{'DE-LU v1':>9s} {'DE-LU v2':>9s} {'Diff':>6s}  "
        f"{'ES v1':>7s} {'ES v2':>7s} {'Diff':>6s}"
    )
    print(hdr)
    print('-' * 72)
    for i in range(len(v1)):
        ts = v1.iloc[i]['timestamp']
        d1 = v1.iloc[i]['DE-LU p50']
        d2 = v2.iloc[i]['DE-LU p50']
        e1 = v1.iloc[i]['ES p50']
        e2 = v2.iloc[i]['ES p50']
        print(
            f'{ts:<28s} {d1:9.2f} {d2:9.2f} {d2-d1:+6.2f}  '
            f'{e1:7.2f} {e2:7.2f} {e2-e1:+6.2f}'
        )

    print()
    print('Overall RMSE between v1 and v2 p50 forecasts:')
    for zone, col in [('DE-LU', 'DE-LU p50'), ('ES', 'ES p50')]:
        rmse = float(np.sqrt(((v2[col] - v1[col]) ** 2).mean()))
        print(f'  {zone}: RMSE = {rmse:.2f} EUR/MWh')

Column-wise mean absolute difference (v2 - v1):
--------------------------------------------------
  DE-LU p025    : MAD=13.86  net_bias=+5.89
  DE-LU p50     : MAD=10.61  net_bias=+9.52
  DE-LU p975    : MAD=26.64  net_bias=+26.46
  ES p025       : MAD=18.38  net_bias=+15.16
  ES p50        : MAD=25.60  net_bias=+24.56
  ES p975       : MAD=43.64  net_bias=+43.64

Row-wise p50 comparison (v1 vs v2):
------------------------------------------------------------------------
Timestamp                     DE-LU v1  DE-LU v2   Diff    ES v1   ES v2   Diff
------------------------------------------------------------------------
2026-05-11T02:00:00+01:00        93.09     98.27  +5.18    59.64   79.19 +19.55
2026-05-11T03:00:00+01:00        89.15     98.13  +8.98    39.59   75.99 +36.40
2026-05-11T04:00:00+01:00        94.24    103.97  +9.73    44.03   75.98 +31.95
2026-05-11T05:00:00+01:00       110.60    113.05  +2.45    49.11   79.79 +30.68
2026-05-11T06:00:00+01:00       117.88    126.94  